**03-Random-Split**  
This notebook runs model training + evaluation pipeline on Scheme A, random cell-level split. This random cell-level split allows cells from the same donor to appear in both train and test sets, potentially leading to optimistic performance estimates.

Structure: We run four models here, with an additional exploration of LMMs.  
1. Filtered dataset based on cell type abundance, logistic regression baseline  
2. Filtered dataset, logistic regression + batch covariate
3. (Baseline) Full dataset, logistic regression  
4. Full dataset, logistic regression + batch covariate
5. Explanation to use Linear Mixed Models (LMMs), as recommended in the intermediate feedback.

In [10]:
import os
from pathlib import Path

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    roc_auc_score, 
    average_precision_score
)
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM

In [11]:
REPO_ROOT = Path("~/scRNA-cross-donor-generalization").expanduser()  # change if needed
os.chdir(REPO_ROOT)

In [ ]:
# helper functions and global settings for running models and evaluating using random split
CELLTYPE_COL = "cell_type"
RANDOM_STATE = 42
TEST_SIZE = 0.2

REPRESENTATIONS = {
    "hvg": "hvg",
    "pca": "X_pca",
    "harmony": "X_harmony",
    "scvi": "X_scVI",
}

def get_batch_matrix(obs, batch_col):
    batch_df = pd.get_dummies(obs[batch_col].astype(str), prefix=batch_col, drop_first=False)
    return batch_df.to_numpy(), batch_df.columns.tolist()

def get_representation(adata, rep_name):
    if rep_name == "hvg":
        X = adata.X
    else:
        X = adata.obsm[rep_name]

    if hasattr(X, "toarray"):
        X = X.toarray()
    else:
        X = np.asarray(X)

    return X

# core function to run logistic regression with random split and optional batch covariates
def run_random_split_logreg(
    adata,
    rep_name,
    celltype_col=CELLTYPE_COL,
    batch_col=None,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
):
    X_rep = get_representation(adata, rep_name)
    y = adata.obs[celltype_col].astype(str)

    counts = y.value_counts()
    keep_labels = counts[counts >= 2].index
    keep_mask = y.isin(keep_labels)

    X_rep = X_rep[keep_mask.to_numpy()]
    y = y[keep_mask].to_numpy()
    obs_sub = adata.obs.loc[keep_mask].copy()

    if batch_col is not None:
        X_batch, batch_feature_names = get_batch_matrix(obs_sub, batch_col)
    else:
        X_batch = None
        batch_feature_names = []

    if X_batch is not None:
        X_train_rep, X_test_rep, X_train_batch, X_test_batch, y_train, y_test = train_test_split(
            X_rep,
            X_batch,
            y,
            test_size=test_size,
            random_state=random_state,
            stratify=y,
        )
    else:
        X_train_rep, X_test_rep, y_train, y_test = train_test_split(
            X_rep,
            y,
            test_size=test_size,
            random_state=random_state,
            stratify=y,
        )

    scaler = StandardScaler()
    X_train_rep = scaler.fit_transform(X_train_rep)
    X_test_rep = scaler.transform(X_test_rep)

    if X_batch is not None:
        X_train = np.hstack([X_train_rep, X_train_batch])
        X_test = np.hstack([X_test_rep, X_test_batch])
    else:
        X_train = X_train_rep
        X_test = X_test_rep

    clf = LogisticRegression(
        max_iter=5000,
        random_state=random_state,
        n_jobs=-1,
    )
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    labels = np.unique(y)
    cm = confusion_matrix(y_test, y_pred, labels=labels)

    macro_f1 = f1_score(y_test, y_pred, average="macro")
    acc = accuracy_score(y_test, y_pred)

    report = classification_report(
        y_test,
        y_pred,
        labels=labels,
        output_dict=True,
        zero_division=0,
    )

    per_class_f1 = pd.DataFrame({
        "cell_type": labels,
        "f1": [report[label]["f1-score"] for label in labels],
        "precision": [report[label]["precision"] for label in labels],
        "recall": [report[label]["recall"] for label in labels],
        "support": [report[label]["support"] for label in labels],
    })

    return {
        "model": clf,
        "scaler": scaler,
        "y_test": y_test,
        "y_pred": y_pred,
        "macro_f1": macro_f1,
        "accuracy": acc,
        "cm": cm,
        "labels": labels,
        "per_class_f1": per_class_f1,
        "n_cells_used": len(y),
        "n_classes_used": len(labels),
        "batch_feature_names": batch_feature_names,
    }


def save_confusion_matrix(cm, labels, out_file, title, normalize=False):
    if normalize:
        row_sums = cm.sum(axis=1, keepdims=True)
        cm_plot = np.divide(
            cm.astype(float),
            row_sums,
            out=np.zeros_like(cm, dtype=float),
            where=row_sums != 0,
        )
    else:
        cm_plot = cm

    fig, ax = plt.subplots(figsize=(12, 10))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm_plot, display_labels=labels)
    disp.plot(
        ax=ax,
        xticks_rotation=90,
        cmap="Blues",
        colorbar=True,
        values_format=None,
    )

    # remove in-cell text for readability
    if disp.text_ is not None:
        for text in disp.text_.ravel():
            if text is not None:
                text.set_visible(False)

    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(out_file, dpi=300, bbox_inches="tight")
    plt.close()

**Case 1: Filtered dataset, Logistic regression without a batch covariate**

In [13]:
# load data, define result directory, and run Scheme A for all representations
adata = sc.read_h5ad("data/adata_filtered_celltypes.h5ad")
results_dir = Path("results/schemeA_case1")
results_dir.mkdir(parents=True, exist_ok=True)

metrics_rows = []

for rep_label, rep_key in REPRESENTATIONS.items():
    print(f"Running Scheme A for {rep_label} ({rep_key})...")

    res = run_random_split_logreg(adata, rep_name=rep_key)

    print(f"  Macro F1: {res['macro_f1']:.4f}")
    print(f"  Accuracy: {res['accuracy']:.4f}")

    # save per-class metrics
    res["per_class_f1"].to_csv(
        results_dir / f"schemeA_{rep_label}_per_class_f1.csv",
        index=False
    )

    pd.DataFrame({
        "y_true": res["y_test"],
        "y_pred": res["y_pred"],
    }).to_csv(
        results_dir / f"schemeA_{rep_label}_predictions.csv",
        index=False
    )

    # save full confusion matrix as csv
    cm_df = pd.DataFrame(res["cm"], index=res["labels"], columns=res["labels"])
    cm_df.to_csv(results_dir / f"schemeA_{rep_label}_confusion_matrix.csv")

    # save normalized confusion matrix plot
    save_confusion_matrix(
        res["cm"],
        res["labels"],
        results_dir / f"schemeA_{rep_label}_confusion_matrix_normalized.png",
        title=f"Scheme A Random Split - {rep_label} (row-normalized)",
        normalize=True,
    )

    # append summary row
    metrics_rows.append({
        "scheme": "random",
        "representation": rep_label,
        "macro_f1": res["macro_f1"],
        "accuracy": res["accuracy"],
        "n_cells_used": res["n_cells_used"],
        "n_classes_used": res["n_classes_used"],
    })

# save metrics table
metrics_case1 = pd.DataFrame(metrics_rows)
metrics_case1 = metrics_case1.sort_values("macro_f1", ascending=False).reset_index(drop=True)
metrics_case1.to_csv(results_dir / "metrics.csv", index=False)

print("\nFinal Scheme A metrics:")
print(metrics_case1)

Running Scheme A for hvg (hvg)...
  Macro F1: 0.7895
  Accuracy: 0.8170
Running Scheme A for pca (X_pca)...
  Macro F1: 0.7898
  Accuracy: 0.8229
Running Scheme A for harmony (X_harmony)...
  Macro F1: 0.7555
  Accuracy: 0.7898
Running Scheme A for scvi (X_scVI)...
  Macro F1: 0.7575
  Accuracy: 0.7918

Final Scheme A metrics:
   scheme representation  macro_f1  accuracy  n_cells_used  n_classes_used
0  random            pca  0.789837  0.822948         10107              13
1  random            hvg  0.789520  0.817013         10107              13
2  random           scvi  0.757488  0.791790         10107              13
3  random        harmony  0.755549  0.789812         10107              13


**Case 2: Filtered dataset, Logistic regression with a batch covariate (one hot based on site)**

In [14]:
# load data, define result directory, and run Scheme A Case 2:
# filtered data + site covariate in logistic regression

adata = sc.read_h5ad("data/adata_filtered_celltypes.h5ad")
results_dir = Path("results/schemeA_case2")
results_dir.mkdir(parents=True, exist_ok=True)

BATCH_COL = "Site"

metrics_rows = []

for rep_label, rep_key in REPRESENTATIONS.items():
    print(f"Running Scheme A Case 2 for {rep_label} ({rep_key}) with batch covariate {BATCH_COL}...")

    res = run_random_split_logreg(
        adata,
        rep_name=rep_key,
        batch_col=BATCH_COL,
    )

    print(f"  Macro F1: {res['macro_f1']:.4f}")
    print(f"  Accuracy: {res['accuracy']:.4f}")

    # save per-class metrics
    res["per_class_f1"].to_csv(
        results_dir / f"schemeA_case2_{rep_label}_per_class_f1.csv",
        index=False
    )

    pd.DataFrame({
        "y_true": res["y_test"],
        "y_pred": res["y_pred"],
    }).to_csv(
        results_dir / f"schemeA_case2_{rep_label}_predictions.csv",
        index=False
    )

    # save full confusion matrix as csv
    cm_df = pd.DataFrame(res["cm"], index=res["labels"], columns=res["labels"])
    cm_df.to_csv(results_dir / f"schemeA_case2_{rep_label}_confusion_matrix.csv")

    # save normalized confusion matrix plot
    save_confusion_matrix(
        res["cm"],
        res["labels"],
        results_dir / f"schemeA_case2_{rep_label}_confusion_matrix_normalized.png",
        title=f"Scheme A Case 2 Random Split + Site Covariate - {rep_label} (row-normalized)",
        normalize=True,
    )

    # append summary row
    metrics_rows.append({
        "scheme": "random_batchcov",
        "representation": rep_label,
        "batch_covariate": BATCH_COL,
        "macro_f1": res["macro_f1"],
        "accuracy": res["accuracy"],
        "n_cells_used": res["n_cells_used"],
        "n_classes_used": res["n_classes_used"],
        "n_batch_features": len(res["batch_feature_names"]),
    })

# save metrics table
metrics_case2 = pd.DataFrame(metrics_rows)
metrics_case2 = metrics_case2.sort_values("macro_f1", ascending=False).reset_index(drop=True)
metrics_case2.to_csv(results_dir / "metrics.csv", index=False)

print("\nFinal Scheme A Case 2 metrics:")
print(metrics_case2)

Running Scheme A Case 2 for hvg (hvg) with batch covariate Site...
  Macro F1: 0.7905
  Accuracy: 0.8185
Running Scheme A Case 2 for pca (X_pca) with batch covariate Site...
  Macro F1: 0.7945
  Accuracy: 0.8254
Running Scheme A Case 2 for harmony (X_harmony) with batch covariate Site...
  Macro F1: 0.7659
  Accuracy: 0.8017
Running Scheme A Case 2 for scvi (X_scVI) with batch covariate Site...
  Macro F1: 0.7652
  Accuracy: 0.7992

Final Scheme A Case 2 metrics:
            scheme representation batch_covariate  macro_f1  accuracy  \
0  random_batchcov            pca            Site  0.794516  0.825420   
1  random_batchcov            hvg            Site  0.790476  0.818497   
2  random_batchcov        harmony            Site  0.765867  0.801682   
3  random_batchcov           scvi            Site  0.765186  0.799209   

   n_cells_used  n_classes_used  n_batch_features  
0         10107              13                 2  
1         10107              13                 2  
2         

**Case 3: Full (Unfiltered) dataset, Logistic regression without a batch covariate**

In [15]:
# load data, define result directory, and run Scheme A for all representations
adata = sc.read_h5ad("data/adata_full_celltypes.h5ad")
results_dir = Path("results/schemeA_case3")
results_dir.mkdir(parents=True, exist_ok=True)

metrics_rows = []

for rep_label, rep_key in REPRESENTATIONS.items():
    print(f"Running Scheme A for {rep_label} ({rep_key})...")

    res = run_random_split_logreg(adata, rep_name=rep_key)

    print(f"  Macro F1: {res['macro_f1']:.4f}")
    print(f"  Accuracy: {res['accuracy']:.4f}")

    # save per-class metrics
    res["per_class_f1"].to_csv(
        results_dir / f"schemeA_{rep_label}_per_class_f1.csv",
        index=False
    )

    pd.DataFrame({
        "y_true": res["y_test"],
        "y_pred": res["y_pred"],
    }).to_csv(
        results_dir / f"schemeA_{rep_label}_predictions.csv",
        index=False
    )

    # save full confusion matrix as csv
    cm_df = pd.DataFrame(res["cm"], index=res["labels"], columns=res["labels"])
    cm_df.to_csv(results_dir / f"schemeA_{rep_label}_confusion_matrix.csv")

    # save normalized confusion matrix plot
    save_confusion_matrix(
        res["cm"],
        res["labels"],
        results_dir / f"schemeA_{rep_label}_confusion_matrix_normalized.png",
        title=f"Scheme A Random Split - {rep_label} (row-normalized)",
        normalize=True,
    )

    # append summary row
    metrics_rows.append({
        "scheme": "random",
        "representation": rep_label,
        "macro_f1": res["macro_f1"],
        "accuracy": res["accuracy"],
        "n_cells_used": res["n_cells_used"],
        "n_classes_used": res["n_classes_used"],
    })

# save metrics table
metrics_case3 = pd.DataFrame(metrics_rows)
metrics_case3 = metrics_case3.sort_values("macro_f1", ascending=False).reset_index(drop=True)
metrics_case3.to_csv(results_dir / "metrics.csv", index=False)

print("\nFinal Scheme A Case 3 metrics:")
print(metrics_case3)

Running Scheme A for hvg (hvg)...
  Macro F1: 0.5089
  Accuracy: 0.7666
Running Scheme A for pca (X_pca)...
  Macro F1: 0.4555
  Accuracy: 0.7724
Running Scheme A for harmony (X_harmony)...
  Macro F1: 0.4804
  Accuracy: 0.7538
Running Scheme A for scvi (X_scVI)...
  Macro F1: 0.5084
  Accuracy: 0.7595

Final Scheme A Case 3 metrics:
   scheme representation  macro_f1  accuracy  n_cells_used  n_classes_used
0  random            hvg  0.508945  0.766608         11289              40
1  random           scvi  0.508365  0.759522         11289              40
2  random        harmony  0.480396  0.753764         11289              40
3  random            pca  0.455523  0.772365         11289              40


**Case 4: Full (Unfiltered) dataset, Logistic regression with a batch covariate**

In [16]:
# load data, define result directory, and run Scheme A Case 4:
# full data + site covariate in logistic regression

adata = sc.read_h5ad("data/adata_full_celltypes.h5ad")
results_dir = Path("results/schemeA_case4")
results_dir.mkdir(parents=True, exist_ok=True)

BATCH_COL = "Site"

metrics_rows = []

for rep_label, rep_key in REPRESENTATIONS.items():
    print(f"Running Scheme A Case 4 for {rep_label} ({rep_key}) with batch covariate {BATCH_COL}...")

    res = run_random_split_logreg(
        adata,
        rep_name=rep_key,
        batch_col=BATCH_COL,
    )

    print(f"  Macro F1: {res['macro_f1']:.4f}")
    print(f"  Accuracy: {res['accuracy']:.4f}")

    # save per-class metrics
    res["per_class_f1"].to_csv(
        results_dir / f"schemeA_case4_{rep_label}_per_class_f1.csv",
        index=False
    )

    pd.DataFrame({
        "y_true": res["y_test"],
        "y_pred": res["y_pred"],
    }).to_csv(
        results_dir / f"schemeA_case4_{rep_label}_predictions.csv",
        index=False
    )

    # save full confusion matrix as csv
    cm_df = pd.DataFrame(res["cm"], index=res["labels"], columns=res["labels"])
    cm_df.to_csv(results_dir / f"schemeA_case4_{rep_label}_confusion_matrix.csv")

    # save normalized confusion matrix plot
    save_confusion_matrix(
        res["cm"],
        res["labels"],
        results_dir / f"schemeA_case4_{rep_label}_confusion_matrix_normalized.png",
        title=f"Scheme A Case 4 Random Split + Site Covariate - {rep_label} (row-normalized)",
        normalize=True,
    )

    # append summary row
    metrics_rows.append({
        "scheme": "random_batchcov",
        "representation": rep_label,
        "batch_covariate": BATCH_COL,
        "macro_f1": res["macro_f1"],
        "accuracy": res["accuracy"],
        "n_cells_used": res["n_cells_used"],
        "n_classes_used": res["n_classes_used"],
        "n_batch_features": len(res["batch_feature_names"]),
    })

# save metrics table
metrics_case4 = pd.DataFrame(metrics_rows)
metrics_case4 = metrics_case4.sort_values("macro_f1", ascending=False).reset_index(drop=True)
metrics_case4.to_csv(results_dir / "metrics.csv", index=False)

print("\nFinal Scheme A Case 4 metrics:")
print(metrics_case4)

Running Scheme A Case 4 for hvg (hvg) with batch covariate Site...
  Macro F1: 0.5086
  Accuracy: 0.7666
Running Scheme A Case 4 for pca (X_pca) with batch covariate Site...
  Macro F1: 0.4724
  Accuracy: 0.7772
Running Scheme A Case 4 for harmony (X_harmony) with batch covariate Site...
  Macro F1: 0.4607
  Accuracy: 0.7640
Running Scheme A Case 4 for scvi (X_scVI) with batch covariate Site...
  Macro F1: 0.5328
  Accuracy: 0.7666

Final Scheme A Case 4 metrics:
            scheme representation batch_covariate  macro_f1  accuracy  \
0  random_batchcov           scvi            Site  0.532800  0.766608   
1  random_batchcov            hvg            Site  0.508551  0.766608   
2  random_batchcov            pca            Site  0.472426  0.777236   
3  random_batchcov        harmony            Site  0.460706  0.763950   

   n_cells_used  n_classes_used  n_batch_features  
0         11289              40                 2  
1         11289              40                 2  
2         

In [19]:
# metric summary across all 4 cases
summary_df = pd.concat([
    metrics_case1.assign(dataset="filtered", batch="no"),
    metrics_case2.assign(dataset="filtered", batch="yes"),
    metrics_case3.assign(dataset="full", batch="no"),
    metrics_case4.assign(dataset="full", batch="yes"),
], ignore_index=True)

summary_df.to_csv("results/tables/schemeA_summary.csv", index=False)
summary_df

,scheme,representation,macro_f1,accuracy,n_cells_used,n_classes_used,dataset,batch,batch_covariate,n_batch_features
0,random,pca,0.789837,0.822948,10107,13,filtered,no,NaN,NaN
1,random,hvg,0.789520,0.817013,10107,13,filtered,no,NaN,NaN
2,random,scvi,0.757488,0.791790,10107,13,filtered,no,NaN,NaN
3,random,harmony,0.755549,0.789812,10107,13,filtered,no,NaN,NaN
4,random_batchcov,pca,0.794516,0.825420,10107,13,filtered,yes,Site,2.0
5,random_batchcov,hvg,0.790476,0.818497,10107,13,filtered,yes,Site,2.0
6,random_batchcov,harmony,0.765867,0.801682,10107,13,filtered,yes,Site,2.0
7,random_batchcov,scvi,0.765186,0.799209,10107,13,filtered,yes,Site,2.0
8,random,hvg,0.508945,0.766608,11289,40,full,no,NaN,NaN
9,random,scvi,0.508365,0.759522,11289,40,full,no,NaN,NaN


**Experimenting with LMM (Linear Mixed Models)**

We additionally explored a mixed-effects logistic regression with donor-level random intercepts for one abundant cell type versus the rest. This was used only as an exploratory sensitivity analysis, since it is binary and evaluated in-sample, and therefore not directly comparable to the main multiclass test-set results.

In [18]:
# Optional exploratory analysis:
# mixed-effects logistic model with donor random intercept
# one abundant cell type vs. all other cells (not directly comparable to main multiclass results)

ADATA_PATH = "data/adata_filtered_celltypes.h5ad"
DONOR_COL = "patient_id"
SITE_COL = "Site"
REP = "X_pca"
TARGET_CELLTYPE = "CD14-positive monocyte"
N_DIMS = 5

# load data and extract low-dimensional representation (first N PCs)
adata = sc.read_h5ad(ADATA_PATH)
X = get_representation(adata, REP)[:, :N_DIMS]
obs = adata.obs.copy()

# build modeling dataframe
pc_cols = [f"{REP}_dim{i+1}" for i in range(N_DIMS)]
df = pd.DataFrame(X, columns=pc_cols)

# binary target: selected cell type vs rest
df["y"] = (obs["cell_type"].astype(str) == TARGET_CELLTYPE).astype(int).to_numpy()

# grouping variables
df[DONOR_COL] = obs[DONOR_COL].astype("category").values   # random effect
df[SITE_COL] = obs[SITE_COL].astype("category").values     # fixed effect

# fixed effects: PCs + site; random intercept: donor
formula = "y ~ " + " + ".join(pc_cols) + f" + C({SITE_COL})"
vc_formulas = {"donor_re": f"0 + C({DONOR_COL})"}

# fit mixed-effects logistic model (variational Bayes)
model = BinomialBayesMixedGLM.from_formula(formula, vc_formulas, df)
result = model.fit_vb()

# in-sample predicted probabilities (using fitted design matrix)
pred_prob = np.clip(
    np.asarray(result.predict(model.exog), dtype=float).ravel(),
    1e-12, 1 - 1e-12
)

# exploratory performance (not comparable to main results)
print(result.summary())
print(f"\nAUROC: {roc_auc_score(df['y'], pred_prob):.4f}")
print(f"AUPRC: {average_precision_score(df['y'], pred_prob):.4f}")

                 Binomial Mixed GLM Results
               Type Post. Mean Post. SD   SD  SD (LB) SD (UB)
-------------------------------------------------------------
Intercept         M   -15.2380   0.1266                      
C(Site)[T.Ncl]    M     0.0192   0.1478                      
X_pca_dim1        M     1.7015   0.0122                      
X_pca_dim2        M    -3.1162   0.0750                      
X_pca_dim3        M    -0.3804   0.0656                      
X_pca_dim4        M    -1.0937   0.0747                      
X_pca_dim5        M    -0.1252   0.1447                      
donor_re          V    -1.2556   0.1499 0.285   0.211   0.385
Parameter types are mean structure (M) and variance structure
(V)
Variance parameters are modeled as log standard deviations

AUROC: 0.9989
AUPRC: 0.9952
